# 01 — GEE Data Pipeline for CarbonLanka

Fetches **real satellite and environmental data** from Google Earth Engine:
- Sentinel-2 NDVI (vegetation health)
- ERA5-Land daily weather (temperature, precipitation, radiation)
- OpenLandMap soil properties (SOC, bulk density, pH, clay%)

Data is fetched for 5 representative Sri Lankan farm locations across different climate zones.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'backend'))

from app.core.gee_client import (
    init_gee, fetch_ndvi, fetch_ndvi_trend,
    fetch_era5_daily, fetch_soilgrids
)
import pandas as pd
import numpy as np
import json

# Initialize GEE
ok = init_gee()
print(f'GEE initialized: {ok}')

In [ ]:
# 5 representative Sri Lankan locations across climate zones
LOCATIONS = [
    {'name': 'Nuwara Eliya', 'lat': 6.9497, 'lng': 80.7891, 'zone': 'tropical_montane', 'crop': 'tea_organic'},
    {'name': 'Kandy',        'lat': 7.2906, 'lng': 80.6337, 'zone': 'tropical_wet',      'crop': 'spice_cinnamon'},
    {'name': 'Ratnapura',    'lat': 6.6828, 'lng': 80.4008, 'zone': 'tropical_wet',      'crop': 'rubber_agroforestry'},
    {'name': 'Gampaha',      'lat': 7.0840, 'lng': 80.0098, 'zone': 'tropical_moist',    'crop': 'coconut_organic'},
    {'name': 'Hambantota',   'lat': 6.1429, 'lng': 81.1212, 'zone': 'tropical_dry',      'crop': 'paddy_rice'},
]
print(f'Fetching data for {len(LOCATIONS)} locations...')

## 1. Sentinel-2 NDVI (Current + 3-Year Trend)

In [ ]:
# Fetch NDVI scores and trends for each location
ndvi_results = []
for loc in LOCATIONS:
    ndvi = fetch_ndvi(loc['lat'], loc['lng'], buffer_m=1000, months_back=12)
    trend = fetch_ndvi_trend(loc['lat'], loc['lng'], buffer_m=1000, years=3)
    ndvi_results.append({
        'district': loc['name'],
        'climate_zone': loc['zone'],
        'crop': loc['crop'],
        'ndvi_mean': ndvi['ndvi_mean'] if ndvi else None,
        'image_count': ndvi.get('image_count', 0) if ndvi else 0,
        'trend_slope': trend['slope'] if trend else None,
        'trend_direction': trend['direction'] if trend else None,
    })
    print(f"  {loc['name']:15s}  NDVI={ndvi['ndvi_mean']:.4f}  trend={trend['direction'] if trend else '?'}")

df_ndvi = pd.DataFrame(ndvi_results)
df_ndvi

## 2. ERA5-Land Daily Weather (365 days)

In [ ]:
# Fetch 365 days of ERA5 weather for each location
all_weather = {}
for loc in LOCATIONS:
    print(f"Fetching ERA5 for {loc['name']}...")
    weather = fetch_era5_daily(loc['lat'], loc['lng'], days=365)
    if weather:
        df = pd.DataFrame(weather)
        df['date'] = pd.to_datetime(df['date'])
        all_weather[loc['name']] = df
        print(f"  Got {len(df)} days  |  T_mean={df['temperature_c'].mean():.1f}C  |  P_total={df['precipitation_mm'].sum():.0f}mm")
    else:
        print(f"  FAILED — no ERA5 data")

print(f'\nSuccessfully fetched weather for {len(all_weather)} locations')

## 3. OpenLandMap Soil Properties

In [ ]:
# Fetch soil properties for each location
soil_results = []
for loc in LOCATIONS:
    soil = fetch_soilgrids(loc['lat'], loc['lng'])
    if soil:
        soil['district'] = loc['name']
        soil_results.append(soil)
        print(f"  {loc['name']:15s}  SOC={soil['soc_g_kg']:.1f} g/kg  pH={soil['ph']:.1f}  clay={soil['clay_pct']:.0f}%")

df_soil = pd.DataFrame(soil_results)
df_soil

## 4. Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# NDVI comparison
ax = axes[0, 0]
bars = ax.bar(df_ndvi['district'], df_ndvi['ndvi_mean'], color=['#2ca02c' if d == 'positive' else '#d62728' if d == 'negative' else '#7f7f7f' for d in df_ndvi['trend_direction']])
ax.set_ylabel('NDVI')
ax.set_title('Sentinel-2 NDVI by District (live from GEE)')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

# Temperature time series
ax = axes[0, 1]
for name, df in all_weather.items():
    ax.plot(df['date'], df['temperature_c'].rolling(7).mean(), label=name, alpha=0.8)
ax.set_ylabel('Temperature (C)')
ax.set_title('ERA5 Temperature (7-day rolling mean)')
ax.legend(fontsize=8)

# Precipitation
ax = axes[1, 0]
for name, df in all_weather.items():
    monthly = df.set_index('date').resample('M')['precipitation_mm'].sum()
    ax.plot(monthly.index, monthly.values, label=name, alpha=0.8)
ax.set_ylabel('Precipitation (mm/month)')
ax.set_title('ERA5 Monthly Precipitation')
ax.legend(fontsize=8)

# Soil properties
ax = axes[1, 1]
x = np.arange(len(df_soil))
w = 0.35
ax.bar(x - w/2, df_soil['soc_g_kg'], w, label='SOC (g/kg)', color='#8B4513')
ax.bar(x + w/2, df_soil['clay_pct'], w, label='Clay (%)', color='#CD853F')
ax.set_xticks(x)
ax.set_xticklabels(df_soil['district'], rotation=45)
ax.set_title('Soil Properties (OpenLandMap via GEE)')
ax.legend()

plt.tight_layout()
plt.savefig('data/gee_overview.png', dpi=150)
plt.show()
print('Saved: data/gee_overview.png')

## 5. Save Data for KGML Training

In [ ]:
# Save weather data as CSV files for each district
for name, df in all_weather.items():
    path = f'data/gee_weather_{name.lower().replace(" ", "_")}.csv'
    df.to_csv(path, index=False)
    print(f'Saved: {path} ({len(df)} days)')

# Save NDVI and soil as JSON
df_ndvi.to_json('data/gee_ndvi_all.json', orient='records', indent=2)
df_soil.to_json('data/gee_soil_all.json', orient='records', indent=2)
print('\nAll GEE data saved to notebooks/data/')